In [1]:
llm_config={"model": "qwen2.5:0.5b-instruct","api_type":"ollama"}

In [2]:
task = '''
        Write a concise but engaging blogpost about
       AI future . Make sure the blogpost is
       within 100 words.
       '''


Create a Writer Agent

In [3]:
import autogen

writer = autogen.AssistantAgent(
    name="Writer",
    system_message="You are a writer. You write engaging and concise " 
        "blogpost (with title) on given topics. You must polish your "
        "writing based on the feedback you receive and give a refined "
        "version. Only return your final work without additional comments.",
    llm_config=llm_config,
)

C:\Anaconda3\envs\agenticAI\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
C:\Anaconda3\envs\agenticAI\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.18) which Google will stop supporting in new releases of google.cloud.resourcemanager_v3 once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.cloud.resourcemanager_v3 past that date.
  warnings.warn(message, FutureWarning)


In [4]:
reply=writer.generate_reply(messages=[{'content':task,'role':'user'}])

In [5]:
print(reply)

{'content': 'AI (Artificial Intelligence) has the potential to transform every industry, from healthcare to finance, education, and transportation. The future of AI is exciting and full of possibilities, but it also brings challenges and ethical concerns. As AI continues to advance, it is crucial to consider how it will impact society and the world around us.', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None}


Adding Reflection:
create a critic agent to reflect on the work of the writer agent

In [7]:
critic = autogen.AssistantAgent(
    name="Critic",
    is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    llm_config=llm_config,
    system_message="You are a critic. You review the work of "
                "the writer and provide constructive "
                "feedback to help improve the quality of the content.",
)

In [8]:
res=critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):


        Write a concise but engaging blogpost about
       AI future . Make sure the blogpost is
       within 100 words.
       

--------------------------------------------------------------------------------
Writer (to Critic):

The future of AI is exciting and transformative. It promises to bring about major changes in various sectors, from healthcare and education to transportation and entertainment. With advances in machine learning and natural language processing, AI is poised to revolutionize how we live, work, and play. The potential of AI is vast, and as we move forward, it's important to keep in mind that ethical and responsible AI is crucial for the future.

--------------------------------------------------------------------------------
Critic (to Writer):

The future of AI is bright and transformative, with advancements in machine learning and natural language processing bringing about major changes in various sectors such as healthcare, education, 

Nested Chats

In [9]:
SEO_reviewer = autogen.AssistantAgent(
    name="SEO_Reviewer",
    llm_config=llm_config,
    system_message="You are an SEO reviewer, known for "
        "your ability to optimize content for search engines, "
        "ensuring that it ranks well and attracts organic traffic. " 
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)

In [10]:
legal_reviewer = autogen.AssistantAgent(
    name="Legal_Reviewer",
    llm_config=llm_config,
    system_message="You are a legal reviewer, known for "
        "your ability to ensure that content is legally compliant "
        "and free from any potential legal issues. "
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)

In [11]:
ethics_reviewer = autogen.AssistantAgent(
    name="Ethics_Reviewer",
    llm_config=llm_config,
    system_message="You are an ethics reviewer, known for "
        "your ability to ensure that content is ethically sound "
        "and free from any potential ethical issues. " 
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role. ",
)

In [12]:
meta_reviewer = autogen.AssistantAgent(
    name="Meta_Reviewer",
    llm_config=llm_config,
    system_message="You are a meta reviewer, you aggragate and review "
    "the work of other reviewers and give a final suggestion on the content.",
)

Orchestrate the nested chats to solve the task

In [13]:
def reflection_message(recipient, messages, sender, config):
    return f'''Review the following content. 
            \n\n {recipient.chat_messages_for_summary(sender)[-1]['content']}'''

review_chats = [
    {
     "recipient": SEO_reviewer, 
     "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}. Here Reviewer should be your role",},
     "max_turns": 1},
    {
    "recipient": legal_reviewer, "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}.",},
     "max_turns": 1},
    {"recipient": ethics_reviewer, "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'reviewer': '', 'review': ''}",},
     "max_turns": 1},
     {"recipient": meta_reviewer, 
      "message": "Aggregrate feedback from all reviewers and give final suggestions on the writing.", 
     "max_turns": 1},
]

In [14]:
critic.register_nested_chats(
    review_chats,
    trigger=writer,
)

In [15]:
res = critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):


        Write a concise but engaging blogpost about
       AI future . Make sure the blogpost is
       within 100 words.
       

--------------------------------------------------------------------------------
Writer (to Critic):

The future of AI is exciting and transformative. It promises to bring about major changes in various sectors, from healthcare and education to transportation and entertainment. With advances in machine learning and natural language processing, AI is poised to revolutionize how we live, work, and play. The potential of AI is vast, and as we move forward, it's important to keep in mind that ethical and responsible AI is crucial for the future.

--------------------------------------------------------------------------------

********************************************************************************
Starting a new chat....

********************************************************************************
Critic (to SEO_Reviewer):

Re

In [16]:
print(res.summary)

Certainly! Here is the revised version of the blogpost:

---

**AI Future: A Comprehensive Look**

The future of Artificial Intelligence (AI) is a rapidly evolving field with profound implications across various sectors. As we move forward, it's essential to keep in mind that ethical and responsible AI is crucial for the future of technology. The future of AI promises transformative changes in healthcare, education, transportation, entertainment, and many more areas, bringing about a significant shift in how we live, work, and play.

**Review 1:**
"The content emphasizes the future of AI and its transformative potential. This is a significant positive aspect."

**Final Suggestion:** "The content highlights the future of AI and its transformative potential. This is a significant positive aspect."

**Review 2:**
"The content emphasizes the future of AI and its transformative potential. This is a significant positive aspect."

**Final Suggestion:** "The content focuses on the future of AI